# Chapter 2 design demo
Projection-inspired and LP-inspired finite-dimensional surrogates on a toy static system.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
for candidate in [cwd, *cwd.parents]:
    src_dir = candidate / 'src'
    if (src_dir / 'mpmgame').exists():
        sys.path.insert(0, str(src_dir))
        break
import numpy as np
import pandas as pd
import mpmgame as mpm
print('using', mpm.__file__)


In [ ]:
G = np.diag([0.75, 0.6, 0.55])
alpha = np.array([[0, 0.25, 0], [0.2, 0, 0.15], [0, 0.2, 0]])
system = mpm.build_contract_system(G, alpha, label='design_base')
Q_init = np.zeros((3,3))
pbar0 = mpm.access_matrix(system, Q_init, 'w2')
v0 = mpm.vulnerability_single_link(system, Q_init, pbar0).value
v0


In [ ]:
proj = mpm.projection_design(system, Q0=Q_init, max_iter=12, access_model='w2', threat_model='single_link', damping=0.6)
iters = pd.DataFrame([{
    'iter': i,
    'surrogate_obj': s.surrogate_obj,
    'measured_vulnerability': s.measured_vulnerability,
    'accepted': s.accepted
} for i, s in enumerate(proj.iterations)])
iters
_ = mpm.plot_projection_convergence(iters)


In [ ]:
lp = mpm.lp_relaxation_design(system, Q_linearization=Q_init, access_model='w2', threat_model='single_link', g_hat=0.25)
lp.success, lp.surrogate_value, lp.true_vulnerability


In [ ]:
vals = [v0, proj.iterations[-1].measured_vulnerability if proj.iterations else v0, lp.true_vulnerability]
labels = ['baseline Q=0', 'projection surrogate', 'LP surrogate']
_ = mpm.plot_design_comparison(labels, vals, title='True measured vulnerability comparison')
print('Note: projection and LP steps are finite-dimensional surrogates/relaxations.')
